# Project Objective

Build an end-to-end forecasting pipeline using the Walmart M5 Forecasting Accuracy dataset. This notebook will inspect, validate, transform, and merge the sales, calendar, and price data for later EDA and modeling.

## Import Libraries

Import the core libraries for file handling and tabular data inspection.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

## Define File Paths

Use notebook-relative paths for raw and processed data folders.

In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root containing data/raw and notebooks")

PROJECT_ROOT = find_project_root()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

calendar_path = RAW_DATA_DIR / "calendar.csv"
prices_path = RAW_DATA_DIR / "sell_prices.csv"
sales_train_validation_path = RAW_DATA_DIR / "sales_train_validation.csv"
sales_train_evaluation_path = RAW_DATA_DIR / "sales_train_evaluation.csv"
sample_submission_path = RAW_DATA_DIR / "sample_submission.csv"

file_paths = {
    "calendar": calendar_path,
    "sell_prices": prices_path,
    "sales_train_validation": sales_train_validation_path,
    "sales_train_evaluation": sales_train_evaluation_path,
    "sample_submission": sample_submission_path,
}

file_status = pd.DataFrame(
    [
        {
            "dataset": name,
            "path": str(path),
            "exists": path.exists(),
            "size_mb": round(path.stat().st_size / (1024 ** 2), 2) if path.exists() else np.nan,
        }
        for name, path in file_paths.items()
    ]
)

file_status

## Load Datasets

Load only the calendar, sell prices, and sales validation datasets for the initial development pipeline. The evaluation file is intentionally not loaded yet.

In [ ]:
required_files = [calendar_path, prices_path, sales_train_validation_path]
missing_files = [path for path in required_files if not path.exists()]

if missing_files:
    raise FileNotFoundError(
        "Missing required raw data files: " + ", ".join(str(path) for path in missing_files)
    )

calendar = pd.read_csv(calendar_path)
prices = pd.read_csv(prices_path)
sales = pd.read_csv(sales_train_validation_path)

datasets = {
    "calendar": calendar,
    "prices": prices,
    "sales": sales,
}

list(datasets.keys())

## Dataset Overview

Preview each loaded dataset without printing the full data.

In [ ]:
for name, df in datasets.items():
    print(f"{name}: shape={df.shape}")
    display(df.head())

In [ ]:
overview = pd.DataFrame(
    [
        {
            "dataset": name,
            "rows": df.shape[0],
            "columns": df.shape[1],
            "memory_mb": round(df.memory_usage(deep=True).sum() / (1024 ** 2), 2),
        }
        for name, df in datasets.items()
    ]
)

overview

In [ ]:
column_names = pd.DataFrame(
    [
        {"dataset": name, "columns": list(df.columns)}
        for name, df in datasets.items()
    ]
)

column_names

## Data Types

Review inferred column data types before making any transformations.

In [ ]:
for name, df in datasets.items():
    print(f"{name} data types")
    display(df.dtypes.to_frame("dtype"))

## Missing Values

Count missing values by column without filling or removing them.

In [ ]:
for name, df in datasets.items():
    print(f"{name} missing values")
    display(df.isna().sum().to_frame("missing_count"))

## Duplicate Records

Count exact duplicate rows in each loaded dataset.

In [ ]:
duplicate_summary = pd.DataFrame(
    [
        {"dataset": name, "exact_duplicate_rows": int(df.duplicated().sum())}
        for name, df in datasets.items()
    ]
)

duplicate_summary

## Dataset Relationships

- `d` connects sales with calendar.
- `wm_yr_wk` connects calendar with prices.
- `store_id` and `item_id` connect sales with prices.

No reshaping or merging is performed in this notebook yet.

In [ ]:
day_columns = [column for column in sales.columns if column.startswith("d_")]

relationship_summary = pd.DataFrame(
    [
        {"metric": "Unique items", "value": sales["item_id"].nunique()},
        {"metric": "Stores", "value": sales["store_id"].nunique()},
        {"metric": "States", "value": sales["state_id"].nunique()},
        {"metric": "Categories", "value": sales["cat_id"].nunique()},
        {"metric": "Departments", "value": sales["dept_id"].nunique()},
        {"metric": "Sales-day columns", "value": len(day_columns)},
        {"metric": "Calendar minimum date", "value": calendar["date"].min()},
        {"metric": "Calendar maximum date", "value": calendar["date"].max()},
        {"metric": "Price minimum week", "value": prices["wm_yr_wk"].min()},
        {"metric": "Price maximum week", "value": prices["wm_yr_wk"].max()},
    ]
)

relationship_summary

## Initial Findings

Use the summaries above to identify file availability, schema consistency, missing values, duplicate records, and key relationship fields before moving into EDA or modeling.

In [ ]:
initial_findings = pd.DataFrame(
    {
        "check": [
            "Required development files available",
            "Evaluation file loaded",
            "Sales data reshaped",
            "Datasets merged",
            "Processed dataset saved",
        ],
        "status": [
            all(path.exists() for path in required_files),
            False,
            False,
            False,
            False,
        ],
        "note": [
            "Calendar, prices, and validation sales files are needed for this notebook.",
            "sales_train_evaluation.csv is reserved for later evaluation work.",
            "Wide sales format is preserved for now.",
            "Relationship keys are documented only.",
            "No output files are written in this notebook.",
        ],
    }
)

initial_findings

# Data Transformation and Merge

Transform a single-store prototype into a long-format table and connect it to calendar and price data. Raw CSV files remain unchanged.

## Data-quality decisions

Convert calendar dates, validate sales values, and check price-key uniqueness before reshaping. Missing event names are normal and mean no special event occurred. Missing sell prices are kept as missing values, not replaced with zero.

In [ ]:
calendar["date"] = pd.to_datetime(calendar["date"])

day_columns = [column for column in sales.columns if column.startswith("d_")]
event_columns = [column for column in calendar.columns if column.startswith("event_")]

sales_values_are_numeric = all(pd.api.types.is_numeric_dtype(sales[column]) for column in day_columns)
sales_minimum = sales[day_columns].min().min()
sales_values_are_non_negative = sales_minimum >= 0

price_key_columns = ["store_id", "item_id", "wm_yr_wk"]
price_keys_are_unique = not prices.duplicated(subset=price_key_columns).any()

quality_decisions = pd.DataFrame(
    [
        {
            "check": "calendar date converted to datetime",
            "result": str(calendar["date"].dtype),
            "decision": "Use datetime dates for time-based validation.",
        },
        {
            "check": "sales values are numeric",
            "result": sales_values_are_numeric,
            "decision": "Proceed only if day-level sales are numeric.",
        },
        {
            "check": "sales values are non-negative",
            "result": sales_values_are_non_negative,
            "decision": "Sales counts should not be negative.",
        },
        {
            "check": "sell price keys are unique",
            "result": price_keys_are_unique,
            "decision": "Use many-to-one validation when merging prices.",
        },
        {
            "check": "missing event values",
            "result": int(calendar[event_columns].isna().sum().sum()) if event_columns else 0,
            "decision": "Keep missing event names unchanged; they indicate no special event.",
        },
        {
            "check": "missing sell prices",
            "result": int(prices["sell_price"].isna().sum()),
            "decision": "Keep missing sell_price values as NaN; do not replace them with zero.",
        },
    ]
)

quality_decisions

## Select a manageable prototype

Use store `CA_1` for pipeline development because transforming all stores at once may require excessive memory. The original full sales dataset remains unchanged.

In [ ]:
identifier_columns = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]

ca1_sales_wide = sales.loc[
    sales["store_id"].eq("CA_1"),
    identifier_columns + day_columns,
].copy()

prototype_summary = pd.DataFrame(
    [
        {"metric": "Full sales rows", "value": sales.shape[0]},
        {"metric": "CA_1 prototype rows", "value": ca1_sales_wide.shape[0]},
        {"metric": "CA_1 day columns", "value": len(day_columns)},
        {"metric": "Full sales unchanged", "value": sales.shape[0] >= ca1_sales_wide.shape[0]},
    ]
)

prototype_summary

## Convert CA_1 sales from wide to long format

Convert `d_` columns into one row per item and sales day. No other stores are transformed.

In [ ]:
import gc

ca1_sales_long = ca1_sales_wide.melt(
    id_vars=identifier_columns,
    value_vars=day_columns,
    var_name="d",
    value_name="sales",
)

rows_after_melt = ca1_sales_long.shape[0]

del ca1_sales_wide
gc.collect()

pd.DataFrame(
    [
        {"metric": "Rows after CA_1 wide-to-long transform", "value": rows_after_melt},
        {"metric": "Columns after transform", "value": ca1_sales_long.shape[1]},
    ]
)

## Merge datasets

Connect long sales to calendar using `d`, then connect sell prices using `store_id`, `item_id`, and `wm_yr_wk`. Row counts are checked before and after each merge.

In [ ]:
rows_before_calendar_merge = ca1_sales_long.shape[0]

ca1_sales_calendar = ca1_sales_long.merge(
    calendar,
    on="d",
    how="left",
    validate="many_to_one",
)

rows_after_calendar_merge = ca1_sales_calendar.shape[0]

del ca1_sales_long
gc.collect()

rows_before_price_merge = ca1_sales_calendar.shape[0]

ca1_sales_merged = ca1_sales_calendar.merge(
    prices,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left",
    validate="many_to_one",
)

rows_after_price_merge = ca1_sales_merged.shape[0]

del ca1_sales_calendar
gc.collect()

merge_row_counts = pd.DataFrame(
    [
        {
            "merge_step": "sales to calendar",
            "rows_before": rows_before_calendar_merge,
            "rows_after": rows_after_calendar_merge,
            "row_count_unchanged": rows_before_calendar_merge == rows_after_calendar_merge,
        },
        {
            "merge_step": "calendar sales to prices",
            "rows_before": rows_before_price_merge,
            "rows_after": rows_after_price_merge,
            "row_count_unchanged": rows_before_price_merge == rows_after_price_merge,
        },
    ]
)

merge_row_counts

## Create data-quality fields

Flag whether a sell price is available, keep missing prices as `NaN`, keep event fields unchanged, and sort the prototype table.

In [ ]:
ca1_sales_merged["price_available"] = ca1_sales_merged["sell_price"].notna().astype("int8")

ca1_sales_merged = ca1_sales_merged.sort_values(["item_id", "date"]).reset_index(drop=True)

ca1_sales_merged[["item_id", "store_id", "date", "sales", "sell_price", "price_available"]].head()

## Validate the transformed data

Review the final CA_1 prototype before saving it for EDA and model development.

In [ ]:
validation_summary = pd.DataFrame(
    [
        {"metric": "Rows", "value": ca1_sales_merged.shape[0]},
        {"metric": "Columns", "value": ca1_sales_merged.shape[1]},
        {"metric": "Minimum date", "value": ca1_sales_merged["date"].min()},
        {"metric": "Maximum date", "value": ca1_sales_merged["date"].max()},
        {"metric": "Unique item count", "value": ca1_sales_merged["item_id"].nunique()},
        {
            "metric": "Duplicate item_id + date rows",
            "value": int(ca1_sales_merged.duplicated(subset=["item_id", "date"]).sum()),
        },
        {"metric": "Sales minimum", "value": ca1_sales_merged["sales"].min()},
        {"metric": "Sales maximum", "value": ca1_sales_merged["sales"].max()},
        {
            "metric": "Memory usage MB",
            "value": round(ca1_sales_merged.memory_usage(deep=True).sum() / (1024 ** 2), 2),
        },
        {"metric": "All records belong to CA_1", "value": ca1_sales_merged["store_id"].eq("CA_1").all()},
    ]
)

validation_summary

In [ ]:
ca1_sales_merged.head()

In [ ]:
ca1_missing_values = ca1_sales_merged.isna().sum().to_frame("missing_count")
ca1_missing_values[ca1_missing_values["missing_count"] > 0]

## Save the result

Save the CA_1 prototype as Parquet to reduce file size and preserve data types. The processed output is ignored by Git through `data/processed/*`.

In [ ]:
processed_ca1_path = PROCESSED_DATA_DIR / "ca1_sales_long.parquet"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

ca1_sales_merged.to_parquet(processed_ca1_path, index=False)

pd.DataFrame(
    [
        {
            "output_path": str(processed_ca1_path),
            "exists": processed_ca1_path.exists(),
            "size_mb": round(processed_ca1_path.stat().st_size / (1024 ** 2), 2),
        }
    ]
)

## Conclusion

The CA_1 sales records were transformed from wide daily columns into a long-format prototype table. Sales connect to calendar through `d`, and prices connect through `store_id`, `item_id`, and `wm_yr_wk`. CA_1 was selected to develop the pipeline with manageable memory use before scaling to more stores. Missing event values were left unchanged because they indicate no special event, and missing sell prices were preserved as `NaN` with a `price_available` flag. This transformed dataset will support EDA and model development.